# Openweathermap API key for live data

In [7]:
import requests

In [ ]:
API_key = "58e655ccae1ee3023aa00d9a9ea778ea"
url = f"https://api.openweathermap.org/data/2.5/weather?lat=44.34&lon=10.99&appid={API_key}"

response = requests.get(url = url)

print(response) # that means my api key has not been activated yet. It may take couple of hours

<Response [401]>


In [ ]:
def get_current_weather_data_of(city: str):
    # It will be done after the api key will be activated. This function will return city name, current temperature, feels like,
    # min temp, max temp, humidity, description and country name.
    pass

# Preprocessing the dataset will be used for training and testing

In [64]:
import polars as pl
from polars import col
pl.Config.set_tbl_cols(n = 10)

polars.config.Config

In [57]:
df = pl.read_csv("weather.csv", ignore_errors = True)
df.head(5)

MinTemp,MaxTemp,WindGustDir,WindGustSpeed,Humidity,Pressure,Temp,RainTomorrow
f64,f64,str,i64,i64,f64,f64,str
8.0,24.3,"""NW""",30,29,1015.0,23.6,"""Yes"""
14.0,26.9,"""ENE""",39,36,1008.4,25.7,"""Yes"""
13.7,23.4,"""NW""",85,69,1007.2,20.2,"""Yes"""
13.3,15.5,"""NW""",54,56,1007.0,14.1,"""Yes"""
7.6,16.1,"""SSE""",50,49,1018.5,15.4,"""No"""


### Rmeoving rows have at least 1 Null value.

In [ ]:
# checking which row has at least 1 null value and which column(s).
print( df.filter(pl.any_horizontal(col('*').is_null()) == True) )

# WindGustDir column has 2 missing values and we can't just fill them with NW/ENE/.. randomly. So cutting those rows.
df = df.drop_nulls(subset = 'WindGustSpeed') # we already know which column has all null values from the previous line.

# rechecking if any row has at least 1 null value.
print( df.filter(pl.any_horizontal(col('*').is_null()) == True) )

shape: (2, 8)
┌─────────┬─────────┬─────────────┬───────────────┬──────────┬──────────┬──────┬──────────────┐
│ MinTemp ┆ MaxTemp ┆ WindGustDir ┆ WindGustSpeed ┆ Humidity ┆ Pressure ┆ Temp ┆ RainTomorrow │
│ ---     ┆ ---     ┆ ---         ┆ ---           ┆ ---      ┆ ---      ┆ ---  ┆ ---          │
│ f64     ┆ f64     ┆ str         ┆ i64           ┆ i64      ┆ f64      ┆ f64  ┆ str          │
╞═════════╪═════════╪═════════════╪═══════════════╪══════════╪══════════╪══════╪══════════════╡
│ -0.1    ┆ 18.0    ┆ NA          ┆ null          ┆ 46       ┆ 1028.7   ┆ 17.4 ┆ No           │
│ 0.8     ┆ 12.2    ┆ NA          ┆ null          ┆ 49       ┆ 1016.8   ┆ 11.2 ┆ No           │
└─────────┴─────────┴─────────────┴───────────────┴──────────┴──────────┴──────┴──────────────┘
shape: (0, 8)
┌─────────┬─────────┬─────────────┬───────────────┬──────────┬──────────┬──────┬──────────────┐
│ MinTemp ┆ MaxTemp ┆ WindGustDir ┆ WindGustSpeed ┆ Humidity ┆ Pressure ┆ Temp ┆ RainTomorrow │
│ ---     ┆ 

### Fixing datatypes

In [110]:
print(df.head(5))
print("\n\n")
print(df['WindGustDir'].value_counts().sort(by = 'count'))
print("\n\n")
print(df.describe())

shape: (5, 8)
┌─────────┬─────────┬─────────────┬───────────────┬──────────┬──────────┬──────┬──────────────┐
│ MinTemp ┆ MaxTemp ┆ WindGustDir ┆ WindGustSpeed ┆ Humidity ┆ Pressure ┆ Temp ┆ RainTomorrow │
│ ---     ┆ ---     ┆ ---         ┆ ---           ┆ ---      ┆ ---      ┆ ---  ┆ ---          │
│ f64     ┆ f64     ┆ cat         ┆ i16           ┆ i64      ┆ f64      ┆ f64  ┆ cat          │
╞═════════╪═════════╪═════════════╪═══════════════╪══════════╪══════════╪══════╪══════════════╡
│ 8.0     ┆ 24.3    ┆ NW          ┆ 30            ┆ 29       ┆ 1015.0   ┆ 23.6 ┆ Yes          │
│ 14.0    ┆ 26.9    ┆ ENE         ┆ 39            ┆ 36       ┆ 1008.4   ┆ 25.7 ┆ Yes          │
│ 13.7    ┆ 23.4    ┆ NW          ┆ 85            ┆ 69       ┆ 1007.2   ┆ 20.2 ┆ Yes          │
│ 13.3    ┆ 15.5    ┆ NW          ┆ 54            ┆ 56       ┆ 1007.0   ┆ 14.1 ┆ Yes          │
│ 7.6     ┆ 16.1    ┆ SSE         ┆ 50            ┆ 49       ┆ 1018.5   ┆ 15.4 ┆ No           │
└─────────┴─────────┴─────

In [65]:
df = df.with_columns(WindGustDir   = col('WindGustDir').cast(pl.Categorical),
                     WindGustSpeed = col('WindGustSpeed').cast(pl.Int16),
                     RainTomorrow  = col('RainTomorrow').cast(pl.Categorical))

df.head(5)

MinTemp,MaxTemp,WindGustDir,WindGustSpeed,Humidity,Pressure,Temp,RainTomorrow
f64,f64,cat,i16,i64,f64,f64,cat
8.0,24.3,"""NW""",30,29,1015.0,23.6,"""Yes"""
14.0,26.9,"""ENE""",39,36,1008.4,25.7,"""Yes"""
13.7,23.4,"""NW""",85,69,1007.2,20.2,"""Yes"""
13.3,15.5,"""NW""",54,56,1007.0,14.1,"""Yes"""
7.6,16.1,"""SSE""",50,49,1018.5,15.4,"""No"""


# Model Training

In [111]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score

from xgboost import  XGBClassifier

### train, test dataset split

In [92]:
le = LabelEncoder()

X = df.select(col('*').exclude('RainTomorrow')) # 2D shape.
y = le.fit_transform( df.select(pl.last()).to_series() ) # 1D shape.
#                     -------- output column ---------

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

### Making Pipeline for XGBoost Classifier

In [99]:
cat_features = ['WindGustDir']
oe = OrdinalEncoder(handle_unknown = "use_encoded_value", unknown_value = -1)

preprocessor = ColumnTransformer(transformers = [("categorical", oe, cat_features)],
                                 remainder    = "passthrough") # keep rest columns as they are.

xgbClf_pipeline = Pipeline(steps = [("preprocessor", preprocessor),
                                    ("classifier",   XGBClassifier(objective = 'binary:logistic',
                                                                   eval_metric = 'logloss',
                                                                   use_label_encoder = False))])

### Best Model searching

In [102]:
parameters = {'classifier__colsample_bytree': [0.4, 0.5, 0.8], 
              'classifier__learning_rate'   : [0.1, 0.2, 0.01],
              'classifier__max_depth'       : [4, 5, 7, 9, 12],
              'classifier__n_estimators'    : [100, 150, 200],
              'classifier__subsample'       : [0.4, 0.5, 0.7],
              'classifier__random_state'    : [42, 50, 62]}


search = GridSearchCV(estimator = xgbClf_pipeline,
                      param_grid = parameters,
                      cv = 5,
                      scoring = 'accuracy',
                      n_jobs = -1,
                      error_score = 'raise')

search.fit(X = X_train, y = y_train)

print(f"Best Accuracy Score = {search.best_score_}.")
print(f"\nBest Hyperparameters values : \n{search.best_params_}.")

best_pipeline = search.best_estimator_ # GridSearchCV already trained this best model with training dataset.

Best Accuracy Score = 0.8694330800701344.

Best Hyperparameters values : 
{'classifier__colsample_bytree': 0.4, 'classifier__learning_rate': 0.1, 'classifier__max_depth': 9, 'classifier__n_estimators': 150, 'classifier__random_state': 42, 'classifier__subsample': 0.4}.


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [09:53:42] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [103]:
type(best_pipeline)

sklearn.pipeline.Pipeline

In [113]:
predict = best_pipeline.predict(X = X_test)

print(predict)

accuracy_score(y_true = y_test, y_pred = predict)

[0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 1 0 0 0 0 0 1 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


0.8082191780821918